In [3]:
TEST = """
rozdział siedemnasty nowy [-zwierzchnik
od-] {+zwierzchnikod+} początku roku szkolnego wiedziano powszechnie że surowy inspektor nazywany w owe czasy rektorem został od obowiązków swych usunięty i że pełni je tylko czasowo czekając na przybycie następcy o powodach usunięcia różne między chłopcami krążyły wieści twierdzono przeważnie że to była kara za zbytnią surowość jakiś knot rozpieszczony przez matkę nigdy palcem przez nikogo nie tknięty jedynaczek po otrzymaniu z rozkazu inspektora dwudziestu [-rózeg-] {+ruzek+} miał ciężko rozchorować się podobno nawet [-umrzeć
osobistości-] {+umrzećosobistości+} ofiary nikt nie umiał dokładnie wskazać sam jednak fakt narodzenia się tej wieści i jej prawdopodobieństwa był bardzo wymowny być może zresztą że sprawa przedstawiała się nierównie [-prościej
inspektor-] {+prościejinspektor+} człowiek stary już mógł [-był-] {+by+} lata
"""

In [4]:
REPLACE = r"\[-([^-]*)-\] \{\+([^\+]*)\+\}"
INSERT = r"\{\+([^\+]*)\+\}"
DELETE = r"\[-([^-]*)-\]"

In [5]:
def process_wdiff_output(text):
    text = text.replace("\n", " ")
    

In [43]:
import re

def get_confusables():
    CONFUSABLES_RAW = {
        "e": ["ę", "e"],
        "ę": ["ę", "e", "en", "em"]
    }
    CONFUSEABLES_SYM = """
    sz ż rz
    t d
    p b
    dź ć
    ą om on oł
    w f
    s z
    ź ś
    dz c
    cz dż drz
    h ch
    k g
    """
    for cfs in CONFUSEABLES_SYM.split("\n"):
        if cfs == "":
            continue
        cfsitems = cfs.strip().split(" ")
        for cfsitem in cfsitems:
            if cfsitem == '':
                continue
            CONFUSABLES_RAW[cfsitem] = cfsitems
    return {x: f"({'|'.join(CONFUSABLES_RAW[x])})" for x in CONFUSABLES_RAW}

def get_confusable_regex(word):
    confusables = get_confusables()
    search_re = re.compile(rf"({'|'.join(sorted(confusables.keys(), key=len, reverse=True))})")

    last_end = 0
    result = ""
    for match in re.finditer(search_re, word):
        start, end = match.span()
        if start != last_end:
            result = result + word[last_end:start]
        result = result + confusables[word[start:end]]
        last_end = end
    result = result + word[last_end:]
    return result

def eq_mod_confusables(a, b):
    a = a.replace(" ", "")
    b = b.replace(" ", "")
    regex = "^" + get_confusable_regex(a) + "$"
    return re.match(regex, b) is not None

def eqnospace(a, b):
    return a.replace(" ", "") == b.replace(" ", "")

def get_replacement(text):
    rep = re.match(REPLACE, text)
    if rep:
        a = rep.group(1)
        b = rep.group(2)
        if eqnospace(a, b):
            return a
        elif eq_mod_confusables(a, b):
            return a
        else:
            return a.replace(" ", "_") + "__" + b.replace(" ", "_")
    elif re.match(INSERT, text) or re.match(DELETE, text):
        return text[2:-2].replace(" ", "") + "__SIL"
    else:
        return text

In [44]:
"{+foo+}"[2:-2]

'foo'

In [41]:
assert get_replacement("[-zwierzchnik od-] {+zwierzchnikod+}") == 'zwierzchnik od'


In [42]:
assert eq_mod_confusables("szczotka", "szczodka") == True
assert eq_mod_confusables("szczotka", "szczodkab") == False
assert eq_mod_confusables("szczotek", "szczodeg") == True